# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
RENAME_MAP ={
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "customer_firstname",
    "cst_lastname": "customer_lastname",
    "cst_marital_status": "customer_marital_status",
    "cst_gndr": "customer_gender",
    "cst_create_date": "create_date"

}

# Reading From Bronze 


In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Data Transformations

##Trimming

In [0]:
from pyspark.sql.types import StringType

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df = df.withColumn("cst_marital_status", F.lower(col("cst_marital_status")))

## Normalization

In [0]:
from pyspark.sql.functions import when

df = (
    df
    .withColumn("cst_marital_status", 
        when(F.lower(col("cst_marital_status")) == "s", "Single")
        .when(F.lower(col("cst_marital_status")) == "m", "Married")
        .otherwise("n/a")
    )
    .withColumn("cst_gndr", 
        when(F.lower(col("cst_gndr")) == "f", "Female")
        .when(F.lower(col("cst_gndr")) == "m", "Male")
        .otherwise("n/a")
    )
)

In [0]:
df.display()


# Renaming The Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Removing null

In [0]:
df = df.dropna()
df.display()

# Write Into Silver Table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.crm_customers")
)